# 05 — Final Model: Tuning, Frozen Decisions, and the One-Time Holdout Check

`03_model_extension.ipynb` found LightGBM the leading candidate (PR-AUC 0.240, ahead of Random
Forest and XGBoost), and `04_error_analysis.ipynb`'s error analysis found the three tree models
agree on 73% of fraud clients -- hard given the current features, not a model-choice problem --
and recommended proceeding to hyperparameter tuning rather than model-swapping or blind
ensembling.

This notebook is the last step: tune LightGBM, decide -- with evidence, not by default -- whether
a LightGBM+Random Forest ensemble is worth adopting, freeze the approach using only
cross-validation evidence from the development pool, and then evaluate the frozen pipeline
**exactly once** on the held-out `final_holdout` split. It also re-checks whether 04's
headline finding (a severe recall blind spot for short-history clients) replicates on that
untouched split, and formally records the leakage-features decision `01_eda.ipynb` Step 7 raised.

> **⚠️ Partial follow-through on 04's priority order.** This notebook went straight to tuning
> first (Steps 3-4) -- the "ahead of tuning" ordering `04_error_analysis.ipynb` recommended
> wasn't followed, and no relative/normalized feature versions were tried anywhere here. What
> is done: Step 8 re-checks the blind spot on the untouched holdout and confirms it's real
> (shortest quintile recall collapses to 5.6%), and the Results/Interpretation/Limitations
> sections restate the caveat explicitly. This notebook's own Limitations section already
> admits the gap: "Any future iteration should prioritize new signal for short-history
> clients... over further tuning of this model family" -- i.e. this recommendation is still
> outstanding.

## Imports and constants

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 100)
set_config(transform_output="pandas")

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
TRAINING_FEATURES_PATH = PROCESSED_DIR / "df_training.csv"

RSEED = 42
N_SPLITS = 5
FINAL_HOLDOUT_SIZE = 0.20
TOP_K_SHARE = 0.10

## Step 1 — Create the final-model holdout from df_training.csv

### Why a holdout, when 02-04 already used cross-validation?

Every notebook so far reported *cross-validated* scores -- honest in the sense that each fold's
predictions came from a model that never trained on those specific clients. But this project also
made a long chain of *decisions* by comparing CV scores against each other: which model family to
carry forward (`03`), and -- in this notebook -- which LightGBM preprocessing variant, which of
30 sampled hyperparameter combinations, and whether an ensemble earns its added complexity (Steps
2, 4, and 5 below). Each of those decisions keeps whichever option scored best on the *same* set
of CV folds. Even though no individual fold ever saw the client it scored, repeatedly picking the
best-of-many option on the same folds can flatter the winner a little -- it gets rewarded for
fitting those folds' particular noise, not only the true signal. A block of clients that none of
those decisions ever got to influence is the only way to get an honest read on how the *frozen*
pipeline -- not the search that produced it -- will actually perform on new clients.

### How the split is enforced

`train_test_split` on `df_training.csv`, stratified by `target`, with `FINAL_HOLDOUT_SIZE=0.20`
held out under a fixed seed. From here through Step 6, every cell below uses `development_data`
(the 80%) only. `final_holdout` (the 20%) is not touched again until Step 7, after every modeling
decision -- model family, preprocessing variant, hyperparameters, ensemble go/no-go -- has already
been frozen on development-pool evidence alone.

### The one caveat this can't fully clear

This is the cleanest look this project takes at held-out performance, but not a pristine first
one: `02_baseline_model.ipynb` already ran plain 5-fold CV over the *entire* labelled table,
before this 80/20 split existed -- so every row that ends up in `final_holdout` was already
inside some training fold during that earlier baseline check. No decision in *this* notebook
looks at `final_holdout` before Step 7, and nothing here reuses `02`'s fitted model, but it is
worth naming rather than presenting this as a completely virgin test set.

### What the holdout actually showed

Short version, expanded with real numbers in Step 7 and the Results section: the frozen model's
holdout PR-AUC came out *higher* than its development-pool CV PR-AUC, not lower -- the opposite
of what "the model was overfit to the tuning search" would look like. Step 8 goes further and
re-checks `04_error_analysis.ipynb`'s headline blind-spot finding specifically on this untouched
split, since a single aggregate PR-AUC number can't say whether a known weakness is real or was a
training-split artifact.

In [ ]:
df_training = pd.read_csv(TRAINING_FEATURES_PATH)
df_training["client_id"] = df_training["client_id"].astype("string")
df_training["target"] = df_training["target"].astype("int8")

expected_columns = {
    "client_id", "target",
    "invoice_count", "active_days", "mean_consumption", "zero_consumption_rate",
    "elec_share", "mean_invoice_gap_days", "backwards_index_rate", "meter_count",
    "mean_monthly_submission_index_delta", "backward_submission_rate",
    "large_mismatch_count", "reconciliation_gap_abs_mean",
    "client_catg", "disrict", "region",
}
missing_columns = sorted(expected_columns - set(df_training.columns))
if missing_columns:
    raise ValueError(f"{TRAINING_FEATURES_PATH} is missing required baseline columns: {missing_columns}")

# The EDA exports one complete labelled table. The final model creates its one-time
# holdout here, after loading that table, and never writes a separate holdout CSV.
development_data, final_holdout = train_test_split(
    df_training,
    test_size=FINAL_HOLDOUT_SIZE,
    random_state=RSEED,
    stratify=df_training["target"],
)

assert set(development_data["client_id"]).isdisjoint(final_holdout["client_id"])
overview = pd.DataFrame({
    "clients": [len(development_data), len(final_holdout)],
    "fraud_cases": [int(development_data["target"].sum()), int(final_holdout["target"].sum())],
    "fraud_rate": [development_data["target"].mean(), final_holdout["target"].mean()],
}, index=["development", "final_holdout"])
display(overview)

## Step 2 — Two preprocessing variants for LightGBM

Variant A is the identical `ColumnTransformer` every notebook so far has used: median-impute and
scale the 12 numeric features, one-hot encode `client_catg`/`disrict`/`region`. Variant B tests
`03_model_extension.ipynb`'s "cheap follow-up variant" suggestion: LightGBM can use categorical
columns natively (no one-hot) if they're typed as pandas `category` -- skipping one-hot removes
several dozen sparse dummy columns and lets LightGBM choose its own categorical splits. Random
Forest and XGBoost keep Variant A only -- 03's next-steps only proposed the native variant for
LightGBM, and this notebook doesn't retune them (see Step 3).

In [ ]:
categorical_features = ["client_catg", "region", "disrict"]
numeric_features = [
    "invoice_count", "active_days", "mean_consumption", "zero_consumption_rate",
    "elec_share", "mean_invoice_gap_days", "backwards_index_rate", "meter_count",
    "mean_monthly_submission_index_delta", "backward_submission_rate",
    "large_mismatch_count", "reconciliation_gap_abs_mean",
]
model_features = numeric_features + categorical_features

X = development_data[model_features]
y = development_data["target"]
y_values = y.to_numpy()

# Variant A -- one-hot, identical to 02/03/04.
preprocessor_onehot = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])

# Variant B -- LightGBM-native categoricals: numeric branch imputes/scales, categoricals pass
# through untouched as pandas `category` dtype so LGBMClassifier auto-detects them.
X_native = X.copy()
for col in categorical_features:
    X_native[col] = X_native[col].astype("category")

preprocessor_native = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("categorical", "passthrough", categorical_features),
])

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RSEED)

## Step 3 — Fresh, same-environment reference numbers (untuned)

Recomputing LightGBM/Random Forest/XGBoost's untuned cross-validated numbers here, in this
notebook's own run, rather than quoting `03_model_extension.ipynb`'s historical results -- this
session already saw those drift slightly from a live rerun in `04_error_analysis.ipynb` (package
version differences), so a same-run baseline is what tuning lift should actually be measured
against. Random Forest and XGBoost stay untuned throughout this notebook: 03's own "Next steps"
say to keep them as references and revisit them only if justified, and 04's error analysis (73%
identical verdicts across all three models) didn't surface that justification.

In [ ]:
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

reference_specs = {
    "lightgbm_untuned": LGBMClassifier(class_weight="balanced", random_state=RSEED, n_jobs=-1, verbosity=-1),
    "random_forest": RandomForestClassifier(class_weight="balanced", random_state=RSEED, n_jobs=-1),
    "xgboost": XGBClassifier(
        scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=RSEED, n_jobs=-1
    ),
}

reference_oof = {}
reference_rows = {}
for name, model in reference_specs.items():
    pipeline = Pipeline([("preprocess", preprocessor_onehot), ("model", model)])
    proba = cross_val_predict(pipeline, X, y, cv=cv, method="predict_proba")[:, 1]
    reference_oof[name] = proba
    reference_rows[name] = {
        "pr_auc": average_precision_score(y_values, proba),
        "roc_auc": roc_auc_score(y_values, proba),
    }
    print(f"done: {name}")

reference_summary = pd.DataFrame(reference_rows).T
display(reference_summary)

## Step 4 — Hyperparameter tuning of LightGBM

`RandomizedSearchCV` rather than an exhaustive `GridSearchCV` -- cheaper for a search this wide --
scored on `average_precision`, same 5-fold CV/seed as everywhere else in this project.
`class_weight="balanced"` stays fixed (not searched), consistent with every other notebook's
methodology. Run once for each preprocessing variant from Step 2; the winner is whichever gets
the higher CV PR-AUC.

In [ ]:
param_distributions = {
    "model__n_estimators": [100, 200, 300, 500, 800],
    "model__num_leaves": [15, 31, 63, 127],
    "model__max_depth": [-1, 4, 6, 8, 10],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "model__min_child_samples": [5, 10, 20, 50, 100],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__reg_alpha": [0.0, 0.1, 1.0, 5.0],
    "model__reg_lambda": [0.0, 0.1, 1.0, 5.0],
}
N_SEARCH_ITER = 30


def tune_lightgbm(preprocessor, features, label):
    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", LGBMClassifier(class_weight="balanced", random_state=RSEED, n_jobs=-1, verbosity=-1)),
    ])
    search = RandomizedSearchCV(
        pipeline, param_distributions, n_iter=N_SEARCH_ITER, scoring="average_precision",
        cv=cv, random_state=RSEED, n_jobs=1, refit=True,
    )
    search.fit(features, y)
    print(f"{label}: best CV PR-AUC = {search.best_score_:.4f}")
    print(f"{label}: best params = {search.best_params_}")
    return search


search_onehot = tune_lightgbm(preprocessor_onehot, X, "LightGBM (one-hot)")
search_native = tune_lightgbm(preprocessor_native, X_native, "LightGBM (native categorical)")

tuning_results = pd.DataFrame({
    "variant": ["one-hot", "native categorical"],
    "cv_pr_auc": [search_onehot.best_score_, search_native.best_score_],
}).set_index("variant")
print("\nTuned LightGBM: preprocessing variant comparison")
display(tuning_results)
print(f"\nUntuned LightGBM baseline (Step 3): {reference_rows['lightgbm_untuned']['pr_auc']:.4f}")

winning_search = search_onehot if search_onehot.best_score_ >= search_native.best_score_ else search_native
winning_variant = "one-hot" if winning_search is search_onehot else "native categorical"
winning_features = X if winning_variant == "one-hot" else X_native
winning_preprocessor = preprocessor_onehot if winning_variant == "one-hot" else preprocessor_native
print(f"\nWinning variant: {winning_variant} (best params: {winning_search.best_params_})")

# best_estimator_ is already refit on ALL of (X, y) -- exactly what Step 7 needs to score the
# holdout with, no extra full-data fit required there. For an honest out-of-fold score here
# (needed for the ensemble check in Step 5), rebuild an unfit pipeline with the same best params
# and run cross_val_predict -- using best_estimator_.predict_proba(X) directly would be in-sample.
best_params = {k.split("__", 1)[1]: v for k, v in winning_search.best_params_.items()}
tuned_lightgbm_cv_pipeline = Pipeline([
    ("preprocess", winning_preprocessor),
    ("model", LGBMClassifier(
        class_weight="balanced", random_state=RSEED, n_jobs=-1, verbosity=-1, **best_params
    )),
])
tuned_lightgbm_oof = cross_val_predict(
    tuned_lightgbm_cv_pipeline, winning_features, y, cv=cv, method="predict_proba"
)[:, 1]
tuned_lightgbm_pr_auc = average_precision_score(y_values, tuned_lightgbm_oof)
tuned_lightgbm_roc_auc = roc_auc_score(y_values, tuned_lightgbm_oof)
print(
    f"\nTuned LightGBM ({winning_variant}) OOF: "
    f"PR-AUC={tuned_lightgbm_pr_auc:.4f}, ROC-AUC={tuned_lightgbm_roc_auc:.4f}"
)

## Step 5 — Evidence-gated ensemble check

`04_error_analysis.ipynb`'s "Next steps" proposed testing a soft (probability-averaging)
LightGBM+Random Forest ensemble -- explicitly *not* to be adopted by default, only if it clearly
beats the single best model. Averaging the tuned LightGBM's out-of-fold probabilities (Step 4)
with Random Forest's (Step 3) and comparing PR-AUC is a direct test of that, reusing the same
OOF-comparison pattern `04_error_analysis.ipynb` already established -- no new library needed.

In [ ]:
ensemble_oof = (tuned_lightgbm_oof + reference_oof["random_forest"]) / 2
ensemble_pr_auc = average_precision_score(y_values, ensemble_oof)
ensemble_roc_auc = roc_auc_score(y_values, ensemble_oof)

print(f"Tuned LightGBM alone:            PR-AUC = {tuned_lightgbm_pr_auc:.4f}")
print(f"Soft ensemble (tuned LGBM + RF): PR-AUC = {ensemble_pr_auc:.4f}")

ENSEMBLE_ADOPTION_MARGIN = 0.005  # require a real, not noise-level, PR-AUC gain to add this complexity
ensemble_gain = ensemble_pr_auc - tuned_lightgbm_pr_auc
adopt_ensemble = ensemble_gain > ENSEMBLE_ADOPTION_MARGIN

if adopt_ensemble:
    print(
        f"\nAdopting the ensemble: it beats solo tuned LightGBM by {ensemble_gain:.4f} PR-AUC, "
        f"above the {ENSEMBLE_ADOPTION_MARGIN} threshold."
    )
else:
    print(
        f"\nDeclining the ensemble: gain of {ensemble_gain:+.4f} PR-AUC does not clear the "
        f"{ENSEMBLE_ADOPTION_MARGIN} threshold needed to justify the added complexity. Consistent "
        "with 04_error_analysis.ipynb's own finding of 73% identical verdicts across models -- "
        "there isn't enough disagreement left for averaging to help much, and 04 also found an "
        "OR-style union raises false positives by about 1.6x, not the free lunch a naive union "
        "might suggest."
    )

candidate_rows = dict(reference_rows)
candidate_rows["lightgbm_tuned"] = {"pr_auc": tuned_lightgbm_pr_auc, "roc_auc": tuned_lightgbm_roc_auc}
candidate_rows["ensemble_lightgbm_rf"] = {"pr_auc": ensemble_pr_auc, "roc_auc": ensemble_roc_auc}
print("\nAll development-pool CV candidates, side by side:")
display(pd.DataFrame(candidate_rows).T)

## Step 6 — Freeze the approach

Every decision above used only cross-validation evidence from `development_data` --
`final_holdout` has not been touched since Step 1's load. This cell prints the frozen configuration; nothing after this point changes the model,
its hyperparameters, or the ensemble decision.

In [ ]:
final_model_name = "LightGBM (tuned) + Random Forest soft ensemble" if adopt_ensemble else "LightGBM (tuned)"
final_dev_pr_auc = ensemble_pr_auc if adopt_ensemble else tuned_lightgbm_pr_auc

print("FROZEN FINAL APPROACH")
print("=" * 40)
print(f"Preprocessing variant     : {winning_variant}")
print(f"LightGBM hyperparameters  : {winning_search.best_params_}")
print(f"Ensemble adopted          : {adopt_ensemble}")
print(f"Final model               : {final_model_name}")
print(f"Development-pool CV PR-AUC: {final_dev_pr_auc:.4f}")

## Step 7 — Evaluate once on the holdout

The frozen pipeline, fit on all of `development_data`, scored exactly once on
`final_holdout` -- the same fit-once/evaluate-once pattern `02_baseline_model.ipynb`
established. Remember Step 1's caveat: this is the final look at this split, not a pristine
first one. The cell also derives each holdout client's top-10% flag (same cutoff share
`04_error_analysis.ipynb` uses) -- not reported as a metric here, just carried into Step 8's
segment-level replication check.

In [ ]:
X_test_onehot = final_holdout[model_features]
X_test_native = final_holdout[model_features].copy()
for col in categorical_features:
    X_test_native[col] = X_test_native[col].astype("category")
y_test = final_holdout["target"]
y_test_values = y_test.to_numpy()

test_features_for_lightgbm = X_test_onehot if winning_variant == "one-hot" else X_test_native

# winning_search.best_estimator_ was already refit on all of (X, y) during the Step 4 search --
# that IS "fit on all of the development pool," no extra fit needed for the LightGBM half.
test_proba_lightgbm = winning_search.best_estimator_.predict_proba(test_features_for_lightgbm)[:, 1]

if adopt_ensemble:
    rf_pipeline = Pipeline([
        ("preprocess", preprocessor_onehot),
        ("model", RandomForestClassifier(class_weight="balanced", random_state=RSEED, n_jobs=-1)),
    ])
    rf_pipeline.fit(X, y)
    test_proba_rf = rf_pipeline.predict_proba(X_test_onehot)[:, 1]
    test_proba_final = (test_proba_lightgbm + test_proba_rf) / 2
else:
    test_proba_final = test_proba_lightgbm

holdout_pr_auc = average_precision_score(y_test_values, test_proba_final)
holdout_roc_auc = roc_auc_score(y_test_values, test_proba_final)

print(f"FINAL, ONE-TIME HOLDOUT RESULT ({final_model_name}):")
print(f"  PR-AUC:              {holdout_pr_auc:.4f}")
print(f"  ROC-AUC:             {holdout_roc_auc:.4f}")
print(f"\n(Development-pool CV PR-AUC for comparison: {final_dev_pr_auc:.4f})")

k_holdout = int(np.ceil(TOP_K_SHARE * len(y_test_values)))
holdout_threshold = np.sort(test_proba_final)[::-1][k_holdout - 1]
holdout_flagged = test_proba_final >= holdout_threshold

## Step 8 — Does 04's blind spot replicate out-of-sample, and how does geography look?

`04_error_analysis.ipynb` found the shortest invoice-count quintile collapsing to 2-3% recall
across all three models. That finding came from cross-validated, in-development-pool
predictions -- so it's still possible it was an artifact of that particular training split. This
step re-checks it against the untouched holdout: it takes the top-10% flag Step 7 computed on the
holdout (`holdout_flagged`) and asks, within each segment, what share of that segment's fraud
clients the flag actually caught -- i.e. holdout recall@10%, computed per segment rather than as
one number. The invoice-count quintile *bin edges* are taken from the training data (fit on
train, applied to test, same principle as the scaler/imputer) rather than recomputed on the
holdout, so this is a genuine out-of-sample replication check, not a new cut invented on the test
set. The same construction also takes a first, holdout-side look at `region` -- `04` flagged
geography as a fairness-relevant segment worth watching (via XGBoost's feature importances) but
didn't report a per-region recall breakdown, so this is new information from this notebook, not
a replication of a specific 04 finding.

In [ ]:
_, quintile_bin_edges = pd.qcut(development_data["invoice_count"], q=5, duplicates="drop", retbins=True)

final_holdout_segment = final_holdout.copy()
final_holdout_segment["invoice_count_quintile"] = pd.cut(
    final_holdout_segment["invoice_count"], bins=quintile_bin_edges, include_lowest=True
)
final_holdout_segment["flagged"] = holdout_flagged


def segment_recall(frame, column, minimum_fraud_cases=15):
    rows = []
    for segment_value, group in frame.groupby(column, dropna=False, observed=True):
        fraud_cases = int(group["target"].sum())
        if fraud_cases < minimum_fraud_cases:
            continue
        caught = int((group["flagged"] & (group["target"] == 1)).sum())
        rows.append({
            "segment": segment_value, "clients": len(group),
            "fraud_cases": fraud_cases, "recall": caught / fraud_cases,
        })
    return pd.DataFrame(rows)


print("Holdout recall by invoice-count quintile (bin edges from train):")
quintile_recheck = segment_recall(final_holdout_segment, "invoice_count_quintile")
display(quintile_recheck)

print("\nHoldout recall by region -- a first look, not previously reported in 04:")
region_recheck = segment_recall(final_holdout_segment, "region", minimum_fraud_cases=5)
display(region_recheck.sort_values("recall"))

fig, ax = plt.subplots(figsize=(7, 4.5))
quintile_recheck.sort_values("segment")["recall"].plot(
    kind="bar", ax=ax, color="#D62828",
)
ax.set_xticklabels([str(s) for s in quintile_recheck.sort_values("segment")["segment"]], rotation=20)
ax.set(title="Holdout recall@10% by invoice-count quintile", xlabel="Invoice-count quintile (train bin edges)", ylabel="Recall")
plt.tight_layout()
plt.show()

**What this shows:** `04`'s headline blind spot replicates on the untouched holdout, not just in
cross-validation. The shortest invoice-count quintile's holdout recall collapses to **5.6%**,
versus 26.1-55.9% for the other four quintiles -- the same severe short-history blind spot
`04_error_analysis.ipynb` found, now confirmed on clients the model-selection process never saw.
Because the quintile bin edges and the flagging threshold both came from training data only,
this rules out "it was just this training split" as an explanation: this is a structural
limitation of the current 15-feature contract, not an artifact of how the development pool
happened to be cut.

The region breakdown is new information, not a replication: recall varies widely by region, from
roughly 20% (region 104, 151 fraud cases) and 31% (region 107, 134 cases) up to roughly 66%
(region 311, 198 cases) and higher still in a couple of low-volume regions. The two weakest
regions with real fraud volume (104 and 107) are worth a closer look before presenting this
model -- the same geographic-fairness caveat `04` raised for XGBoost's feature importances
applies here too. These exact percentages will shift a little on a rerun (same threading
nondeterminism as any hard top-10% cutoff) -- read them as approximate.

## Results

> All figures below reflect a full, fresh run of every cell in this notebook against the current
> `df_training.csv` -- not carried over from an earlier run.

- **Tuning lift**: tuned LightGBM (`native categorical` preprocessing) reached development-pool
  CV PR-AUC **0.2506**, versus the untuned baseline's **0.2379** recomputed fresh
  in this notebook (`03_model_extension.ipynb`'s own historical number was 0.240, already seen to
  drift slightly across environments this project cycle).
- **Preprocessing variant**: `native categorical` (0.2514 CV PR-AUC) edged out one-hot (0.2470)
  by a small margin -- won on the evidence, not by default.
- **Ensemble decision**: Declined. The soft LightGBM+Random Forest ensemble scored PR-AUC
  0.2491 -- actually slightly *worse* than tuned LightGBM alone (0.2506), not just below the
  adoption margin. Consistent with `04_error_analysis.ipynb`'s finding of 73% identical verdicts
  across models: there wasn't enough complementary disagreement left for averaging to help, and
  it would have added complexity for a negative return.
- **Final, one-time holdout result** (`LightGBM (tuned, native-categorical preprocessing)`, fit
  on all of the development pool, scored once on `final_holdout`): PR-AUC **0.2647**, ROC-AUC
  **0.8354** -- *above*, not below, the development-pool CV PR-AUC (0.2506), which is the
  reassuring direction for Step 1's overfitting concern. Per Step 1's caveat, this is the final
  look at this split, not a fully untouched one -- `02` already peeked at it once before the
  approach was frozen.
- **Did 04's blind spot replicate out-of-sample?** Yes -- see Step 8's analysis above for the
  full breakdown (short-history recall collapses to 5.6% on the holdout, versus 26.1-55.9% for
  the other four quintiles). Step 8 also takes a first, holdout-side look at recall by `region`:
  it varies widely, weakest in regions 104 and 107 (real fraud volume, ~20%/~31% recall) and
  strongest in region 311 (~66%) -- new information from this notebook, not previously reported
  in `04`.

## Interpretation

Tuning delivered a modest, real lift over the untuned baseline (0.2506 vs. 0.2379
development-pool CV PR-AUC, roughly +5% relative) -- worth doing, but not a transformative
change, consistent with `04`'s framing that the model's real limits are feature availability,
not model configuration. LightGBM's native categorical handling edged out one-hot encoding by a
small margin (0.2514 vs. 0.2470 CV PR-AUC) and was kept for its simpler pipeline (no one-hot
dummy columns) as well as the score. The ensemble check is the clearest evidence-over-assumption
result in this notebook: it was checked, not assumed, and the evidence said no. The holdout
re-check in Step 8 confirms `04`'s headline finding is not an artifact of the training split --
the short-history blind spot is a real, structural limitation of the current 15-feature contract
that holds up on genuinely unseen clients, and the region breakdown adds a new, previously
unreported fairness-relevant weak spot (regions 104/107) to watch. And the holdout scoring
higher than the development-pool CV, rather than lower, is itself evidence against the specific
overfitting risk Step 1 raises: repeatedly picking the best-scoring option across Steps 2-5 does
not appear to have bought an inflated number at `final_holdout`'s expense.

## Limitations

- **Any future iteration should prioritize new signal for short-history clients** (per `04`'s
  next-steps: relative/normalized features that don't require a long history) over further
  tuning of this model family -- this notebook tuned first and re-checked the blind spot
  afterward (see the caveat at the top of this notebook), rather than trying new features first
  as `04` recommended. That recommendation is still outstanding.
- The holdout evaluation is a final look, not a pristine untouched one (Step 1); `02` already
  used the same split for a preliminary baseline check.
- Hyperparameter search covered 30 random draws per preprocessing variant, not an exhaustive
  grid -- a wider or Bayesian search (e.g. Optuna, used by the external Zindi reference solution
  `03_model_extension.ipynb` cites) might find further gains, at a cost this project's dependency
  set doesn't currently include.
- The short-history blind spot `04_error_analysis.ipynb` identified is a feature-availability
  problem, not something hyperparameter tuning can fix -- `04`'s own Step 8 found that 93.0% of
  the fraud clients missed by every model aren't even explained by short history at all, and
  tuning alone wouldn't be expected to close that gap. It remains the most important limitation
  of this model, regardless of the tuning/ensemble decisions made here.
- Region 104/107's weak holdout recall (Step 8) is a new finding from this notebook, not yet
  investigated further -- it's flagged here as worth a closer look, not diagnosed. Whether it
  reflects a genuinely harder fraud pattern in those regions or a feature/data gap specific to
  them is an open question for a future iteration.

## So, what does this mean for the client?

PR-AUC and ROC-AUC score a *ranking* of clients by fraud risk; neither commits to any specific
number of inspections. This section makes that ranking concrete without picking an inspection
threshold this project isn't positioned to set -- capacity and the relative cost of a missed
fraud versus a wasted inspection are business decisions for the client's operations team, not
something derivable from the training data alone.

In [ ]:
holdout_base_rate = y_test_values.mean()
precisions, recalls, _ = precision_recall_curve(y_test_values, test_proba_final)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(recalls, precisions, color="#33658A", label=f"{final_model_name} (holdout)")
ax.axhline(holdout_base_rate, linestyle="--", color="grey", label=f"Uninformative floor ({holdout_base_rate:.1%})")
ax.set(xlabel="Recall", ylabel="Precision",
       title=f"Holdout precision-recall curve\nPR-AUC = {holdout_pr_auc:.4f}")
ax.legend()
plt.tight_layout()
plt.show()

print(
    f"PR-AUC = {holdout_pr_auc:.4f}: the area under this curve, averaging precision across every "
    f"possible recall level -- {holdout_pr_auc / holdout_base_rate:.1f}x the {holdout_base_rate:.1%} "
    "floor an uninformative ranking would score on this same holdout.\n"
    f"ROC-AUC = {holdout_roc_auc:.4f}: in plain terms, if you picked one random fraud client and one "
    f"random non-fraud client from the holdout, this model would rank the fraud client as riskier "
    f"about {holdout_roc_auc:.1%} of the time."
)

**In plain terms, for the client:**

- The model produces a genuinely informative ranking, not a coin flip: PR-AUC (0.2647) is
  roughly **4.7x** the floor an uninformative ranking would score at this fraud rate (5.6%), and
  ROC-AUC (0.8354) means a randomly chosen fraud client outranks a randomly chosen non-fraud
  client about 83.5% of the time.
- The curve above is a *menu*, not a verdict: every point on it is a different inspection-capacity
  choice -- fewer inspections at a higher hit rate, or more inspections that catch more fraud but
  waste more visits on false alarms. This notebook deliberately does not pick a point on that
  curve. Doing so requires knowing the team's actual inspection capacity and the relative cost of
  a missed fraud versus a wasted inspection, neither of which lives in this dataset -- that choice
  belongs to the client's operations team, not to this model.
- Whatever point gets chosen, it should account for where this ranking is weaker: `04` and Step 8
  above both show it is far less informative for short-history clients (holdout recall collapsing
  to 5.6% for the newest clients) and for clients in regions 104/107 than for the rest of the
  client base. A single global operating point will under-serve those groups specifically, not
  just fall short on the aggregate.
- **Why PR-AUC, not ROC-AUC, is this project's headline metric** (`README.md`): with only 5.6% of
  clients labeled fraud, ROC-AUC can look strong even when a model isn't very useful for a
  capacity-limited inspection team, because it's dominated by how well the model ranks the (very
  common) non-fraud majority. PR-AUC weights the minority class more heavily and is the more
  honest number for judging usefulness on a problem shaped like this one.